# 13. Control experiment: is the resolution difference between sources the cause?

**Question.** Non-durian images that are reduced and compressed get labelled as a disease. Is this because the model learned the shortcut "low image quality means disease" (H1), or simply because it never saw degraded images (H0)?

**Part A (CPU, no training).** Measure the quality of the images the model actually sees (256 px cache) by source, test how well quality alone separates disease from `other`, and build `other` images whose effective resolution matches the disease images.

**Part B (GPU).** Train EfficientNet-B0 with three seeds per arm. All arms share the training code and differ only in how image quality is handled.

| Arm | Change to the training images | H1 predicts (other→disease when degraded) | H0 predicts |
|---|---|---|---|
| `base` | none | high | high |
| `match_other` | reduce `other` images so their effective-resolution distribution matches the disease images (taken from the data, not from the test protocol) | **much lower** | about the same |
| `match_other_jpeg` | as above, plus JPEG at quality 60–90 | much lower | about the same |
| `qaug_disease` | quality augmentation on disease images only | **no better, or worse** | somewhat better |
| `qaug_other` | quality augmentation on `other` images only | lower | lower |
| `qaug` | quality augmentation on all images (reference) | lower | lower |

**Pre-specified criteria** (fixed before any result, at 112 px; metric = other→disease on the original test set; gap = `base` − `qaug`):

- **P1** manipulation check: after matching, the KS distance in effective resolution between disease and `other` training images is below 0.10.
- **P2** `match_other` closes at least 50% of the gap with a CI for (match − base) below zero: clear support for H1. 20–50%: partial. Below 20% or a CI that crosses zero: no support.
- **P3** `qaug_disease` closes less than 20% of the gap (or makes it worse): consistent with H1. At least 50%: contradicts H1.
- **P4** `match_other` does not raise disease-image accuracy at 112 px by more than 2 points over `base` (a larger rise would indicate general robustness rather than removal of a shortcut).

**Running.** Part A runs on a Colab CPU. Part B: Kaggle GPU with the private dataset attached and Internet on. `base` and `qaug` reuse the `best.pt` files from notebooks 08/03 when present, and the notebook checks that they reproduce the stored numbers; otherwise they are retrained.

In [ ]:
# 0. Settings
import os, sys, io, json, time, math, random, shutil, zipfile, subprocess, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image

IN_KAGGLE = bool(os.environ.get('KAGGLE_KERNEL_RUN_TYPE'))
IN_COLAB = (not IN_KAGGLE) and Path('/var/colab/hostname').exists()
MOCK = os.environ.get('CAUSAL_MOCK') == '1'      # synthetic data, for testing the code only

RUN_PART_A = True
RUN_PART_B = True          # needs a GPU; skipped otherwise
MODEL_NAME = 'efficientnet_b0.ra_in1k'
PRETRAINED = True
SEEDS = [42, 1, 2]
ARMS = ['base', 'match_other', 'qaug_disease', 'qaug_other', 'match_other_jpeg', 'qaug']   # in order of priority
EPOCHS, PATIENCE, BATCH = 30, 8, 64
LR, WD, LABEL_SMOOTH = 1e-3, 0.05, 0.1
LEVELS, JPEG_Q = [224, 160, 112, 80], 70
IMG_SIZE = 224
# quality augmentation, same values as notebooks 03/08
Q_P_DOWNSCALE, Q_SCALE, Q_P_JPEG, Q_JPEG, Q_P_NOISE = 0.6, (0.25, 1.0), 0.6, (30, 96), 0.3
REUSE_REF = True           # reload base/qaug best.pt from 08/03 instead of retraining
SAVE_CKPT = True
N_BOOT = 2000
MATCH_SEED = 1234          # matched 'other' images are identical across training seeds

if MOCK:
    MODEL_NAME, PRETRAINED = 'mobilenetv3_small_050', False
    SEEDS, EPOCHS, PATIENCE, BATCH, N_BOOT = [42, 1], 2, 8, 16, 200

print(f'platform: kaggle={IN_KAGGLE} colab={IN_COLAB} mock={MOCK}')

In [ ]:
# 1. Locate data
def find_data_dir(root):
    c = [p.parent for p in Path(root).rglob('cache256.npy') if (p.parent / 'metadata.csv').exists()]
    if not c:
        raise FileNotFoundError(f'cache256.npy + metadata.csv not found under {root}')
    c.sort(key=lambda p: (('v11' in str(p).lower()) or ('v1_1' in str(p).lower()), str(p)))   # prefer v1
    return c[0]

if IN_KAGGLE:
    DATA_DIR = find_data_dir('/kaggle/input')
    OUT_DIR = Path('/kaggle/working/causal')
    CACHE_PATH = DATA_DIR / 'cache256.npy'
elif IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = Path('/content/drive/MyDrive/durian_work')
    OUT_DIR = DATA_DIR / 'results' / 'causal'
    CACHE_PATH = Path('/content/cache256.npy')          # local copy; Drive reads drop out
    if not CACHE_PATH.exists():
        print('copying cache256.npy to local disk (1.2 GB)')
        shutil.copy(DATA_DIR / 'cache256.npy', CACHE_PATH)
else:
    DATA_DIR = Path(os.environ.get('DURIAN_DATA_DIR', '.'))
    OUT_DIR = Path(os.environ.get('DURIAN_OUT_DIR', './causal'))
    CACHE_PATH = DATA_DIR / 'cache256.npy'
OUT_DIR.mkdir(parents=True, exist_ok=True)

meta_all = pd.read_csv(DATA_DIR / 'metadata.csv')
X = np.load(CACHE_PATH, mmap_mode='r')
assert len(meta_all) == X.shape[0], f'metadata has {len(meta_all)} rows but the cache has {X.shape[0]}'
assert X.shape[1:] == (256, 256, 3) and X.dtype == np.uint8, X.shape
meta_all['row'] = np.arange(len(meta_all))

# With the v1.1 dataset attached, orchard rows (group starting with orchard_) are kept out of training and used as a field test set.
is_orch = meta_all['group'].astype(str).str.startswith('orchard_') if 'group' in meta_all else pd.Series(False, index=meta_all.index)
meta = meta_all[~is_orch].reset_index(drop=True)
FIELD = {}
if is_orch.any():
    FIELD['orchard_rows'] = np.stack([X[r] for r in meta_all.loc[is_orch, 'row']])
    print(f'{int(is_orch.sum())} orchard rows removed from training and kept as a field test set')
for p in sorted(Path(DATA_DIR).rglob('orchard_cache256.npy'))[:1]:
    if 'orchard_rows' not in FIELD:
        FIELD['orchard_npy'] = np.load(p)
print('DATA_DIR =', DATA_DIR, '| OUT_DIR =', OUT_DIR, '| v1 images =', len(meta), '| field sets =', {k: len(v) for k, v in FIELD.items()})
if not MOCK and len(meta) != 6051:
    print(f'WARNING: expected 6,051 v1 images, found {len(meta)}. Check the attached dataset.')

def load_classes():
    labels = sorted(meta['label'].unique())
    for p in sorted(Path(DATA_DIR).rglob('classes.json')):
        try:
            j = json.load(open(p, encoding='utf-8'))
            if isinstance(j, dict):
                j = j.get('classes', j)
            if isinstance(j, dict):
                if all(str(k).isdigit() for k in j):
                    j = [j[k] for k in sorted(j, key=int)]
                else:
                    j = [k for k, _ in sorted(j.items(), key=lambda kv: kv[1])]
            if sorted(j) == labels:
                return list(j)
        except Exception as e:
            print('could not read', p, ':', e)
    print('no matching classes.json; classes sorted alphabetically')
    return labels

CLASSES = load_classes()
C = len(CLASSES); OTHER = CLASSES.index('other')
meta['y'] = meta['label'].map({c: i for i, c in enumerate(CLASSES)}).astype(int)
meta['kind'] = np.where(meta['label'] == 'other', 'other', 'disease')
if 'source' not in meta: meta['source'] = meta['kind']
if 'group' not in meta: meta['group'] = meta['row'].astype(str)
SPLIT = {s: meta[meta['split'] == s] for s in ['train', 'val', 'test']}
IDX = {s: SPLIT[s]['row'].values for s in SPLIT}              # cache row indices (as IDX in 03/08)
y_all = np.full(len(meta_all), -1); y_all[meta['row'].values] = meta['y'].values
_cj = next(iter(sorted(Path(DATA_DIR).rglob('classes.json'))), None)
REF_DIR = _cj.parent if _cj is not None else None             # earlier results folder (efficientnet_b0__base__s42, ...)
print({s: len(d) for s, d in SPLIT.items()}, '| classes', C, '| other in train/test =',
      int((SPLIT['train'].kind == 'other').sum()), int((SPLIT['test'].kind == 'other').sum()))

In [ ]:
# 2. Quality measures and degradation
GRID = np.array([0.125, 0.1875, 0.25, 0.3125, 0.375, 0.4375, 0.5, 0.625, 0.75, 0.875])
TAU, MIN_REF = 0.25, 1.0

def jpeg(im, q):
    b = io.BytesIO(); im.save(b, 'JPEG', quality=int(q)); b.seek(0)
    return Image.open(b).convert('RGB')

def eff_res(arr):
    """Effective resolution (0.125-1). Reduce and enlarge the image at scale s and find where detail starts to be lost.
    An image that is already soft (upscaled from a small one) loses nothing until s falls below its true resolution.
    Normalised by the image's own error at s = 0.125; images that are too flat return NaN."""
    g = Image.fromarray(np.asarray(arr)).convert('L')
    g0 = np.asarray(g, np.float32)
    errs = np.empty(len(GRID), np.float32)
    for i, s in enumerate(GRID):
        n = max(8, int(round(256 * s)))
        errs[i] = np.abs(g0 - np.asarray(g.resize((n, n), Image.BICUBIC).resize((256, 256), Image.BICUBIC), np.float32)).mean()
    if errs[0] < MIN_REF:
        return np.nan
    c = errs / errs[0]
    below = np.where(c < TAU)[0]
    if len(below) == 0: return 1.0
    j = below[0]
    if j == 0: return float(GRID[0])
    return float(GRID[j-1] + (c[j-1] - TAU) / (c[j-1] - c[j] + 1e-9) * (GRID[j] - GRID[j-1]))

def lap_var(arr):
    g = np.asarray(Image.fromarray(np.asarray(arr)).convert('L'), np.float32)
    l = 4 * g[1:-1, 1:-1] - g[:-2, 1:-1] - g[2:, 1:-1] - g[1:-1, :-2] - g[1:-1, 2:]
    return float(l.var())

def degrade_res(arr, s, q=None):
    n = max(16, int(256 * s))
    im = Image.fromarray(np.asarray(arr)).resize((n, n), Image.BILINEAR)       # same resampling as the evaluation protocol
    if q is not None: im = jpeg(im, q)
    return np.asarray(im.resize((256, 256), Image.BICUBIC))

def match_image(arr, target, q=None):
    """Scale s at which eff_res of the image is approximately target (bisection)."""
    lo, hi = 0.10, 1.0
    for _ in range(10):
        mid = (lo + hi) / 2
        e = eff_res(degrade_res(arr, mid, q))
        if np.isnan(e) or e < target: lo = mid
        else: hi = mid
    s = (lo + hi) / 2
    out = degrade_res(arr, s, q)
    return out, s

def degrade(a, s=160, q=70):                      # evaluation protocol, copied verbatim from 03/08
    im = Image.fromarray(a).resize((s, s), Image.BILINEAR)
    buf = io.BytesIO(); im.save(buf, 'JPEG', quality=q)
    return np.asarray(Image.open(buf).convert('RGB').resize((256, 256), Image.BICUBIC))

# standard levels (names as in 03/08), plus variants that separate JPEG from resizing
STD_LEVELS = ['original'] + [f'{L}px' for L in LEVELS]
VARIANTS = STD_LEVELS + ['jpeg_only'] + [f'{L}px_nojpeg' for L in LEVELS[1:]]

def apply_variant(a, name):
    a = np.asarray(a)
    if name == 'original': return a
    if name == 'jpeg_only': return np.asarray(jpeg(Image.fromarray(a), JPEG_Q))
    L = int(name.split('px')[0])
    if name.endswith('_nojpeg'):
        return np.asarray(Image.fromarray(a).resize((L, L), Image.BILINEAR).resize((256, 256), Image.BICUBIC))
    return degrade(a, L, JPEG_Q)

In [ ]:
# 3. Part A: image quality by source, and quality-matched 'other' images
from scipy.stats import ks_2samp
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import roc_auc_score, f1_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

QFILE = OUT_DIR / 'quality_per_image.csv'
if QFILE.exists() and len(pd.read_csv(QFILE)) == len(meta):
    q = pd.read_csv(QFILE); print('using stored quality measurements', QFILE)
else:
    t0 = time.time(); e, l = [], []
    for i, r in enumerate(meta['row']):
        a = X[r]; e.append(eff_res(a)); l.append(lap_var(a))
        if (i + 1) % 1000 == 0: print(f'  quality {i+1}/{len(meta)} ({time.time()-t0:.0f}s)')
    q = meta[['row', 'label', 'kind', 'source', 'split']].copy()
    q['eff_res'], q['lap_var'] = e, l
    q.to_csv(QFILE, index=False)
meta['eff_res'], meta['lap_var'] = q['eff_res'].values, q['lap_var'].values

def summ(d):
    return pd.Series({'n': len(d), 'eff_median': d.eff_res.median(), 'eff_q25': d.eff_res.quantile(.25), 'eff_q75': d.eff_res.quantile(.75),
                      'pct_eff_below_0.5': 100 * (d.eff_res < 0.5).mean(), 'lapvar_median': d.lap_var.median(),
                      'pct_flat_nan': 100 * d.eff_res.isna().mean()})
by_source = meta.groupby(['kind', 'source']).apply(summ)
by_source.to_csv(OUT_DIR / 'quality_by_source.csv')
meta.groupby('label').apply(summ).to_csv(OUT_DIR / 'quality_by_class.csv')
print(by_source.round(3).to_string())

# quantile mapping: the 'other' image at rank q gets the eff_res of the disease images at quantile q
tr = SPLIT['train'].copy()
tr['eff_res'], tr['lap_var'] = meta.loc[tr.index, 'eff_res'], meta.loc[tr.index, 'lap_var']
dis = tr[(tr.kind == 'disease') & tr.eff_res.notna()]
oth = tr[(tr.kind == 'other') & tr.eff_res.notna()].copy()
rng = np.random.default_rng(MATCH_SEED)
oth = oth.iloc[rng.permutation(len(oth))]                      # shuffle first to break ties (many sharp images have eff = 1.0)
oth['q'] = (oth.eff_res.rank(method='first') - 0.5) / len(oth)
oth = oth.sort_index()
oth['target'] = np.quantile(dis.eff_res.values, oth.q.values)
oth['jpeg_q'] = rng.integers(60, 91, len(oth))

MATCHED = {'match_other': {}, 'match_other_jpeg': {}}       # row -> uint8 256 image
rec = []
for _, r in oth.iterrows():
    a = np.asarray(X[int(r.row)])
    need = r.target < r.eff_res - 0.02
    if need:
        out, s = match_image(a, r.target); MATCHED['match_other'][int(r.row)] = out
    else:
        out, s = a, 1.0
    if need:
        outj, sj = match_image(a, r.target, int(r.jpeg_q))
    else:
        outj, sj = degrade_res(a, 1.0, int(r.jpeg_q)), 1.0
    MATCHED['match_other_jpeg'][int(r.row)] = outj
    rec.append(dict(row=int(r.row), source=r.source, eff_before=r.eff_res, target=r.target, changed=bool(need), scale=s,
                    eff_after=eff_res(out), lap_after=lap_var(out), scale_jpeg=sj, eff_after_jpeg=eff_res(outj), lap_after_jpeg=lap_var(outj), jpeg_q=int(r.jpeg_q)))
mrec = pd.DataFrame(rec); mrec.to_csv(OUT_DIR / 'matching_per_image.csv', index=False)

# manipulation check (P1), and how well quality alone separates disease from 'other'
def auc_quality(eff_o, lap_o):
    f = np.r_[np.c_[dis.eff_res, np.log1p(dis.lap_var)], np.c_[eff_o, np.log1p(lap_o)]]
    f = np.where(np.isnan(f), np.nanmedian(f, 0), f)
    yb = np.r_[np.zeros(len(dis)), np.ones(len(eff_o))]
    cv = StratifiedKFold(5, shuffle=True, random_state=0)
    out = {}
    for name, cols in [('eff_res', [0]), ('eff_res+lap_var', [0, 1])]:
        p = cross_val_predict(make_pipeline(StandardScaler(), LogisticRegression(class_weight='balanced', max_iter=1000)), f[:, cols], yb, cv=cv, method='predict_proba')[:, 1]
        out['auc_' + name] = roc_auc_score(yb, p)
    return out

chk = []
for name, ec, lc in [('before', oth.eff_res.values, oth.lap_var.values), ('match_other', mrec.eff_after.values, mrec.lap_after.values),
                     ('match_other_jpeg', mrec.eff_after_jpeg.values, mrec.lap_after_jpeg.values)]:
    ok = ~np.isnan(ec)
    chk.append(dict(state=name, ks_eff=ks_2samp(dis.eff_res.values, ec[ok]).statistic, eff_median_disease=dis.eff_res.median(), eff_median_other=np.nanmedian(ec),
                    pct_other_changed=100 * mrec.changed.mean() if name != 'before' else 0.0, **auc_quality(ec, lc)))
match_check = pd.DataFrame(chk); match_check.to_csv(OUT_DIR / 'matching_check.csv', index=False)
print('\nManipulation check (KS near 0 = matched; AUC 0.5 = quality does not separate disease from other):')
print(match_check.round(3).to_string(index=False))
P1 = bool(match_check.loc[match_check.state == 'match_other', 'ks_eff'].iloc[0] < 0.10)
print('P1 (KS < 0.10):', P1)
if match_check.ks_eff.iloc[0] < 0.10:
    print('WARNING: before matching, disease and other images hardly differ in effective resolution, so the resolution hypothesis is weak from the start. See jpeg_only/nojpeg in part B.')

In [ ]:
# 3b. Part A figures
import matplotlib
matplotlib.use('Agg') if not (IN_COLAB or IN_KAGGLE) else None
import matplotlib.pyplot as plt

def ecdf(ax, v, **k):
    v = np.sort(np.asarray(v)[~np.isnan(v)])
    if len(v): ax.step(v, np.arange(1, len(v) + 1) / len(v), where='post', **k)

# where the test protocol levels sit on the same axis (150 random test images)
samp = SPLIT['test'].sample(min(150, len(SPLIT['test'])), random_state=0)['row']
lvl_eff = {v: np.nanmedian([eff_res(np.asarray(apply_variant(X[r], v))) for r in samp]) for v in STD_LEVELS}
pd.Series(lvl_eff, name='eff_median').to_csv(OUT_DIR / 'eval_level_eff.csv')

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for (k, s), d in tr.groupby(['kind', 'source']):
    ecdf(ax[0], d.eff_res.values, label=f'{k}: {s} (n={len(d)})', lw=2 if k == 'disease' else 1.3, ls='-' if k == 'disease' else '--')
for v, e in lvl_eff.items():
    ax[0].axvline(e, color='gray', lw=0.6, ls=':'); ax[0].text(e, 1.02, v, rotation=90, fontsize=7, ha='center', va='bottom')
ax[0].set(xlabel='effective resolution (fraction of 256 px)', ylabel='cumulative fraction of training images', title='(a) Image quality by source')
ax[0].legend(fontsize=7, loc='upper left')
ecdf(ax[1], dis.eff_res.values, label='disease', lw=2, color='C3')
ecdf(ax[1], oth.eff_res.values, label='other, original', lw=1.5, color='C0', ls='--')
ecdf(ax[1], mrec.eff_after.values, label='other, matched', lw=1.5, color='C2')
ax[1].set(xlabel='effective resolution (fraction of 256 px)', title='(b) Manipulation check'); ax[1].legend(fontsize=8)
plt.tight_layout()
for ext in ['png', 'pdf']: fig.savefig(OUT_DIR / f'fig_quality_by_source.{ext}', dpi=300)
plt.show() if (IN_COLAB or IN_KAGGLE) else plt.close(fig)

# 'other' images before and after matching, for visual inspection
ch = mrec[mrec.changed].sort_values('scale').iloc[np.linspace(0, max(0, mrec.changed.sum() - 1), 6).astype(int)] if mrec.changed.any() else mrec.iloc[:0]
if len(ch):
    fig, ax = plt.subplots(2, len(ch), figsize=(2.2 * len(ch), 4.6))
    ax = np.atleast_2d(ax).reshape(2, -1)
    for i, (_, r) in enumerate(ch.iterrows()):
        ax[0, i].imshow(X[int(r.row)]); ax[0, i].set_title(f'eff {r.eff_before:.2f}', fontsize=8)
        ax[1, i].imshow(MATCHED['match_other'][int(r.row)]); ax[1, i].set_title(f'→ {r.eff_after:.2f} (s={r.scale:.2f})', fontsize=8)
        ax[0, i].axis('off'); ax[1, i].axis('off')
    plt.tight_layout(); fig.savefig(OUT_DIR / 'fig_matching_examples.png', dpi=150)
    plt.show() if (IN_COLAB or IN_KAGGLE) else plt.close(fig)
print('part A figures saved | test levels on the eff_res axis:', {k: round(float(v), 2) for k, v in lvl_eff.items()})

In [ ]:
# 4. Part B: training setup (functions copied from 03/08; only TrainDS takes an arm)
try:
    import torch
    HAS_GPU = torch.cuda.is_available()
except ImportError:
    HAS_GPU = False
DO_B = RUN_PART_B and (HAS_GPU or MOCK)
if RUN_PART_B and not DO_B:
    print('no GPU: part B skipped (part A is done). Run part B on a Kaggle GPU.')

if DO_B:
    try:
        import timm
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'timm'], check=False); import timm
    import torchvision.transforms.v2 as T
    from torch.utils.data import Dataset, DataLoader
    from sklearn.metrics import accuracy_score
    dev = 'cuda' if HAS_GPU else 'cpu'
    MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
    NUM_WORKERS = 0 if MOCK else (4 if IN_KAGGLE else 2)
    XR = np.load(CACHE_PATH)                                    # whole cache in RAM

    def rand_quality(a, rng):
        if rng.random() < Q_P_DOWNSCALE:
            s = max(16, int(256 * rng.uniform(*Q_SCALE)))
            im = Image.fromarray(a).resize((s, s), Image.BILINEAR)
            a = np.asarray(im.resize((256, 256), rng.choice([Image.BILINEAR, Image.BICUBIC, Image.NEAREST])))
        if rng.random() < Q_P_JPEG:
            buf = io.BytesIO(); Image.fromarray(a).save(buf, 'JPEG', quality=int(rng.integers(*Q_JPEG)))
            a = np.asarray(Image.open(buf).convert('RGB'))
        if rng.random() < Q_P_NOISE:
            a = np.clip(a.astype(np.float32) + rng.normal(0, rng.uniform(2, 8), a.shape), 0, 255).astype(np.uint8)
        return a

    tf_train = T.Compose([
        T.RandomResizedCrop(IMG_SIZE, scale=(0.5, 1.0), antialias=True),
        T.RandomHorizontalFlip(), T.RandomVerticalFlip(), T.RandomRotation(20),
        T.ColorJitter(0.35, 0.35, 0.25, 0.02),
        T.RandomApply([T.GaussianBlur(5, (0.1, 2.0))], p=0.2),
        T.ToDtype(torch.float32, scale=True), T.Normalize(MEAN, STD)])
    tf_eval = T.Compose([T.Resize((IMG_SIZE, IMG_SIZE), antialias=True),
                         T.ToDtype(torch.float32, scale=True), T.Normalize(MEAN, STD)])

    def use_qaug(arm, y):
        return arm == 'qaug' or (arm == 'qaug_other' and y == OTHER) or (arm == 'qaug_disease' and y != OTHER)

    class TrainDS(Dataset):
        def __init__(self, idx, arm):
            self.idx, self.arm, self.over = np.asarray(idx), arm, MATCHED.get(arm, {})
        def __len__(self): return len(self.idx)
        def __getitem__(self, k):
            i = int(self.idx[k]); y = int(y_all[i])
            rng = np.random.default_rng(torch.randint(0, 2**31 - 1, (1,)).item())
            a = self.over[i] if i in self.over else XR[i]        # match_*: quality-matched 'other' image (fixed across epochs)
            if use_qaug(self.arm, y): a = rand_quality(a, rng)
            return tf_train(torch.from_numpy(np.array(a)).permute(2, 0, 1)), y

    @torch.no_grad()
    def logits_of(model, arrs):
        model.eval(); out = []
        for s in range(0, len(arrs), BATCH):
            x = torch.stack([tf_eval(torch.from_numpy(np.array(a)).permute(2, 0, 1)) for a in arrs[s:s + BATCH]])
            with torch.autocast(dev, enabled=(dev == 'cuda')):
                out.append(model(x.to(dev, non_blocking=True)).float().cpu())
        return torch.cat(out)

    def ece(probs, y, n_bins=15):
        conf, pred = probs.max(1), probs.argmax(1); acc = (pred == y); e = 0.0
        edges = np.linspace(0, 1, n_bins + 1)
        for lo, hi in zip(edges[:-1], edges[1:]):
            m = (conf > lo) & (conf <= hi)
            if m.any(): e += m.mean() * abs(acc[m].mean() - conf[m].mean())
        return float(e)

    def fit_temperature(logits, y):
        logT = torch.zeros(1, requires_grad=True)
        opt = torch.optim.LBFGS([logT], lr=0.1, max_iter=200); yt = torch.as_tensor(y)
        def closure():
            opt.zero_grad(); loss = torch.nn.functional.cross_entropy(logits / logT.exp(), yt); loss.backward(); return loss
        opt.step(closure)
        return float(logT.detach().exp())

    def metrics(p, y):
        pred = p.argmax(1); o = y == OTHER
        return dict(acc=100 * accuracy_score(y, pred), macro_f1=100 * f1_score(y, pred, average='macro'),
                    other_as_disease=100 * float((pred[o] != OTHER).mean()), disease_as_other=100 * float((pred[~o] == OTHER).mean()),
                    acc_disease=100 * float((pred[~o] == y[~o]).mean()), ece=100 * ece(p, y))

    t0 = time.time()
    TEST_ARRS = {v: [apply_variant(XR[i], v) for i in IDX['test']] for v in VARIANTS}
    VAL_ARRS = [XR[i] for i in IDX['val']]; VAL_DEG = [degrade(XR[i], 112) for i in IDX['val']]
    FIELD_ARRS = {k: {v: [apply_variant(x, v) for x in a] for v in STD_LEVELS} for k, a in FIELD.items()}
    print(f'{len(VARIANTS)} evaluation variants prepared ({time.time()-t0:.0f}s) | device {dev}')

In [ ]:
# 5. Train and evaluate
def make_model():
    return timm.create_model(MODEL_NAME, pretrained=PRETRAINED, num_classes=C, drop_rate=0.2).to(dev)

def evaluate_and_save(model, out, arm, seed, info):
    yt = y_all[IDX['test']]
    T_ = fit_temperature(logits_of(model, VAL_ARRS), y_all[IDX['val']])
    rob, probs = [], {}
    for v in VARIANTS:
        p = (logits_of(model, TEST_ARRS[v]) / T_).softmax(1).numpy(); probs[v] = p.astype(np.float32)
        rob.append(dict(variant=v, **metrics(p, yt)))
    field = {k: {v: 100 * float((logits_of(model, a).argmax(1).numpy() != OTHER).mean()) for v, a in d.items()} for k, d in FIELD_ARRS.items()}
    pd.DataFrame(rob).to_csv(out / 'robustness.csv', index=False)
    np.savez_compressed(out / 'test_probs.npz', idx=IDX['test'], y=yt, **probs)
    (out / 'result.json').write_text(json.dumps(dict(arm=arm, seed=seed, model=MODEL_NAME, temperature=round(T_, 3), train_n=int(len(IDX['train'])),
        torch_version=torch.__version__, timm_version=timm.__version__, field_non_other_pct=field, robustness=rob, **info), indent=1))
    r = {d['variant']: d for d in rob}
    print(f'{out.name} [{info["source"]}]: acc {r["original"]["acc"]:.2f} | other->disease original/112/80 = '
          f'{r["original"]["other_as_disease"]:.1f}/{r["112px"]["other_as_disease"]:.1f}/{r["80px"]["other_as_disease"]:.1f}')
    return r

def reuse_reference(arm, seed, out):
    # base/qaug already trained in 08/03: evaluate the stored best.pt with this code and check that it reproduces the stored numbers
    if not (REUSE_REF and REF_DIR is not None and arm in ('base', 'qaug')): return False
    ref = REF_DIR / f"{MODEL_NAME.split('.')[0]}__{arm}__s{seed}"
    if not ((ref / 'best.pt').exists() and (ref / 'robustness.csv').exists()): return False
    model = make_model(); model.load_state_dict(torch.load(ref / 'best.pt', map_location=dev))
    r = evaluate_and_save(model, out, arm, seed, dict(source='reused_checkpoint', ref_dir=str(ref)))
    old = pd.read_csv(ref / 'robustness.csv').set_index('level')
    d_acc = max(abs(r[l]['acc'] - 100 * old.loc[l, 'acc']) for l in STD_LEVELS if l in old.index)
    d_oad = max(abs(r[l]['other_as_disease'] - 100 * old.loc[l, 'other_as_disease']) for l in STD_LEVELS if l in old.index)
    ok = d_acc <= 0.35 and d_oad <= 2.7           # about 3 of 888 images / 2 of 77 (GPU and AMP nondeterminism)
    print(f'   check against stored results: max diff acc {d_acc:.2f} pts, other->disease {d_oad:.2f} pts -> ' + ('match' if ok else 'MISMATCH: evaluation code differs from 03/08, stop here'))
    j = json.loads((out / 'result.json').read_text()); j['ref_check'] = dict(max_abs_acc=d_acc, max_abs_other_as_disease=d_oad, ok=bool(ok))
    (out / 'result.json').write_text(json.dumps(j, indent=1))
    return True

def train_one(arm, seed):
    out = OUT_DIR / f'{arm}__s{seed}'
    if (out / 'result.json').exists():
        print(f'skip {out.name} (already done)'); return
    out.mkdir(parents=True, exist_ok=True)
    if reuse_reference(arm, seed, out): return
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    tr = IDX['train']
    counts = np.bincount(y_all[tr], minlength=C)
    w = torch.tensor(len(tr) / (C * counts), dtype=torch.float32, device=dev)
    crit = torch.nn.CrossEntropyLoss(weight=w, label_smoothing=LABEL_SMOOTH)
    model = make_model()
    dl = DataLoader(TrainDS(tr, arm), batch_size=BATCH, shuffle=True, num_workers=NUM_WORKERS,
                    pin_memory=(dev == 'cuda'), drop_last=True, persistent_workers=NUM_WORKERS > 0)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=EPOCHS * len(dl), pct_start=0.1)
    scaler = torch.amp.GradScaler(enabled=(dev == 'cuda'))
    best, best_ep, hist, t0 = -1, 0, [], time.time()
    yv = y_all[IDX['val']]
    for ep in range(EPOCHS):
        model.train(); tot = 0
        for x, yb in dl:
            x, yb = x.to(dev, non_blocking=True), yb.to(dev, non_blocking=True)
            with torch.autocast(dev, enabled=(dev == 'cuda')):
                loss = crit(model(x), yb)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            tot += loss.item() * len(yb)
        f_clean = f1_score(yv, logits_of(model, VAL_ARRS).argmax(1).numpy(), average='macro')
        f_deg = f1_score(yv, logits_of(model, VAL_DEG).argmax(1).numpy(), average='macro')
        score = (f_clean + f_deg) / 2
        hist.append(dict(epoch=ep + 1, train_loss=tot / len(dl.dataset), val_f1_clean=f_clean, val_f1_112px=f_deg))
        print(f'  [{arm} s{seed}] ep {ep + 1:2d} | loss {hist[-1]["train_loss"]:.3f} | val F1 clean {f_clean:.4f} | 112px {f_deg:.4f}')
        if score > best:
            best, best_ep = score, ep + 1; torch.save(model.state_dict(), out / 'best.pt')
        elif ep + 1 - best_ep >= PATIENCE:
            print('  early stop'); break
    del dl
    model.load_state_dict(torch.load(out / 'best.pt', map_location=dev))
    pd.DataFrame(hist).to_csv(out / 'history.csv', index=False)
    evaluate_and_save(model, out, arm, seed, dict(source='trained', best_epoch=best_ep, train_minutes=round((time.time() - t0) / 60, 1)))
    if not SAVE_CKPT: (out / 'best.pt').unlink()

if DO_B:
    todo = [(a, s) for s in SEEDS for a in ARMS]          # all arms of the first seed first
    print(f'{len(todo)} runs: {ARMS} x seed {SEEDS} | reference results: {REF_DIR}')
    for a, s in todo:
        train_one(a, s)

In [ ]:
# 6. Summary and pre-specified criteria
def load_results():
    rows = []
    for p in sorted(OUT_DIR.glob('*__s*/result.json')):
        j = json.load(open(p))
        for d in j['robustness']:
            rows.append(dict(arm=j['arm'], seed=j['seed'], source=j.get('source'), **d))
    return pd.DataFrame(rows)

def boot_diff(arm, ref, variant, B=N_BOOT):
    """(arm - ref) of other->disease on the 'other' test images, paired by seed, cluster bootstrap over image groups."""
    seeds = [s for s in SEEDS if (OUT_DIR / f'{arm}__s{s}/test_probs.npz').exists() and (OUT_DIR / f'{ref}__s{s}/test_probs.npz').exists()]
    if not seeds: return None
    d = 0
    for s in seeds:
        za, zr = np.load(OUT_DIR / f'{arm}__s{s}/test_probs.npz'), np.load(OUT_DIR / f'{ref}__s{s}/test_probs.npz')
        o = za['y'] == OTHER
        d = d + ((za[variant][o].argmax(1) != OTHER).astype(float) - (zr[variant][o].argmax(1) != OTHER).astype(float))
    d = d / len(seeds)
    grp = SPLIT['test'].loc[SPLIT['test']['y'].values == OTHER, 'group'].astype(str).values
    gs = pd.Series(d).groupby(grp).agg(['sum', 'count']).values
    rng = np.random.default_rng(0); idx = rng.integers(0, len(gs), (B, len(gs)))
    bs = 100 * gs[idx, 0].sum(1) / gs[idx, 1].sum(1)
    return dict(diff=100 * d.mean(), lo=np.percentile(bs, 2.5), hi=np.percentile(bs, 97.5), n_seeds=len(seeds), n_images=len(d), n_groups=len(gs))

res = load_results() if DO_B or any(OUT_DIR.glob('*__s*/result.json')) else pd.DataFrame()
if len(res):
    res.to_csv(OUT_DIR / 'causal_runs.csv', index=False)
    agg = res.groupby(['arm', 'variant'])[['acc', 'acc_disease', 'other_as_disease', 'disease_as_other', 'ece']].agg(['mean', 'std', 'count'])
    agg.to_csv(OUT_DIR / 'causal_summary.csv')
    show = ['original', 'jpeg_only', '224px', '160px', '112px', '80px', '112px_nojpeg', '80px_nojpeg']
    arms_done = [a for a in ARMS if a in res.arm.unique()]
    for m, title in [('other_as_disease', 'other->disease (%)'), ('acc_disease', 'accuracy on disease images (%)')]:
        t = res.pivot_table(index='arm', columns='variant', values=m, aggfunc=['mean', 'std'])
        tab = pd.DataFrame({v: [f'{t["mean"][v][a]:.1f}±{0 if np.isnan(t["std"][v][a]) else t["std"][v][a]:.1f}' for a in arms_done] for v in show}, index=arms_done)
        print(f'\n{title}, mean±SD over seeds (n seeds = {res.groupby("arm").seed.nunique().to_dict()})'); print(tab.to_string())
        tab.to_csv(OUT_DIR / f'table_causal_{m}.csv')

    mean = res.groupby(['arm', 'variant'])[['other_as_disease', 'acc_disease']].mean()
    verdict = []
    for lv in ['112px', '80px']:
        if ('base', lv) not in mean.index: continue
        b = mean.loc[('base', lv), 'other_as_disease']
        gap = b - mean.loc[('qaug', lv), 'other_as_disease'] if ('qaug', lv) in mean.index else np.nan
        for a in arms_done:
            if a == 'base': continue
            bd = boot_diff(a, 'base', lv) or {}
            v = mean.loc[(a, lv), 'other_as_disease']
            verdict.append(dict(level=lv, arm=a, base=b, arm_value=v, diff=bd.get('diff'), ci_lo=bd.get('lo'), ci_hi=bd.get('hi'),
                                gap_base_minus_qaug=gap, gap_closed_pct=100 * (b - v) / gap if gap and gap > 0 else np.nan,
                                d_acc_disease=mean.loc[(a, lv), 'acc_disease'] - mean.loc[('base', lv), 'acc_disease'], n_seeds=bd.get('n_seeds')))
    verdict = pd.DataFrame(verdict); verdict.to_csv(OUT_DIR / 'causal_verdict.csv', index=False)
    print('\nDifference from base (arm - base, points, 95% cluster-bootstrap CI):'); print(verdict.round(2).to_string(index=False))

    def get(a, col, lv='112px'):
        r = verdict[(verdict.arm == a) & (verdict.level == lv)]
        return float(r[col].iloc[0]) if len(r) and pd.notna(r[col].iloc[0]) else None
    print('\n===== Pre-specified criteria (112 px) =====')
    print('P1 manipulation check (KS < 0.10):', P1)
    if not P1: print('P1 not met: matching failed, so P2/P4 for match_other cannot be interpreted')
    gc, hi = get('match_other', 'gap_closed_pct'), get('match_other', 'ci_hi')
    if gc is not None and hi is not None:
        lab = 'supports H1' if (gc >= 50 and hi < 0) else ('partial support' if (gc >= 20 and hi < 0) else 'does not support H1')
        print(f'P2 match_other closed {gc:.0f}% of the gap, upper CI {hi:.1f} -> {lab}')
    gd = get('qaug_disease', 'gap_closed_pct')
    if gd is not None:
        print(f'P3 qaug_disease closed {gd:.0f}% of the gap -> ' + ('consistent with H1' if gd < 20 else ('contradicts H1' if gd >= 50 else 'inconclusive')))
    da = get('match_other', 'd_acc_disease')
    if da is not None:
        print(f'P4 match_other changed disease-image accuracy by {da:+.1f} pts -> ' + ('met (not general robustness)' if da <= 2 else 'not met'))
    print('result source:', res.drop_duplicates(['arm', 'seed']).groupby('arm').source.agg(lambda x: '/'.join(sorted(set(map(str, x))))).to_dict())
    if res.groupby('arm').seed.nunique().min() < 3: print('WARNING: not all arms have 3 seeds; do not report these numbers')
    for a, ref, tol in [('base', 42.9, 10), ('qaug', 9.5, 6)]:
        if (a, '112px') in mean.index and not MOCK and abs(mean.loc[(a, '112px'), 'other_as_disease'] - ref) > tol:
            print(f'WARNING: {a} @112 = {mean.loc[(a, "112px"), "other_as_disease"]:.1f}% differs from 08/03 ({ref}%) by more than {tol} pts. Check resampling and augmentation.')

    # figure
    lv = STD_LEVELS
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    for i, a in enumerate(arms_done):
        for k, m in enumerate(['other_as_disease', 'acc_disease']):
            mu = [agg.loc[(a, v), (m, 'mean')] for v in lv]; sd = [np.nan_to_num(agg.loc[(a, v), (m, 'std')]) for v in lv]
            ax[k].errorbar(range(len(lv)), mu, yerr=sd, marker='o', capsize=2, label=a, lw=2 if a in ('base', 'match_other') else 1.2)
    for k, (yl, tt) in enumerate([('non-durian images predicted as disease (%)', '(a) Rejection failure'), ('accuracy on disease images (%)', '(b) Disease recognition')]):
        ax[k].set_xticks(range(len(lv))); ax[k].set_xticklabels(lv); ax[k].set(ylabel=yl, title=tt, xlabel='test image degradation')
    ax[0].legend(fontsize=8); plt.tight_layout()
    for ext in ['png', 'pdf']: fig.savefig(OUT_DIR / f'fig_causal.{ext}', dpi=300)
    plt.show() if (IN_COLAB or IN_KAGGLE) else plt.close(fig)
else:
    print('no part B results yet')

In [ ]:
# 7. Bundle results (without best.pt)
ZIP = OUT_DIR.parent / 'causal_results.zip'
with zipfile.ZipFile(ZIP, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT_DIR.rglob('*')):
        if p.is_file() and p.suffix != '.pt':
            z.write(p, Path('causal') / p.relative_to(OUT_DIR))
print('saved', ZIP, f'({ZIP.stat().st_size/1e6:.1f} MB)')